# Sentiment Analysis of IMDB Movie Reviews

**Problem Statement:**

Predict positive and negative reviews based on sentiments using different classification models (ML + DL + Transformer).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import re

from bs4 import BeautifulSoup

import nltk
from nltk.tokenize.toktok import ToktokTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

# Seq DL libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, SimpleRNN, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Transformers libraries
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
import torch

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
lemmatizer = WordNetLemmatizer()
tokenizer  = ToktokTokenizer()

## 2. Load Data

**Dataset:** [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

In [ ]:
import kagglehub

path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [ ]:
import os

imdb_data = pd.read_csv(os.path.join(path, 'IMDB Dataset.csv'))

In [ ]:
imdb_data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 3. EDA

In [ ]:
imdb_data.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [ ]:
imdb_data['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [ ]:
print('Duplicate reviews:', imdb_data.duplicated(subset=['review']).sum())

Duplicate reviews: 418


In [ ]:
imdb_data.drop_duplicates(subset=['review'], inplace=True)
print('Rows after deduplication:', len(imdb_data))

Rows after deduplication: 49582


In [ ]:
imdb_data['sentiment'].value_counts()

,count
sentiment,
positive,24884
negative,24698


## 4. Preprocessing

### Cleaning Functions

In [ ]:
contractions_dict = {
    "can't":    "can not",
    "cannot":   "can not",
    "won't":    "will not",
    "wouldn't": "would not",
    "shouldn't":"should not",
    "couldn't": "could not",
    "don't":    "do not",
    "doesn't":  "does not",
    "didn't":   "did not",
    "isn't":    "is not",
    "aren't":   "are not",
    "wasn't":   "was not",
    "weren't":  "were not",
    "haven't":  "have not",
    "hasn't":   "has not",
    "hadn't":   "had not",
    "it's":     "it is",
    "i'm":      "i am",
    "you're":   "you are",
    "they're":  "they are",
    "we're":    "we are",
    "i've":     "i have",
    "you've":   "you have",
    "we've":    "we have",
    "they've":  "they have",
    "i'll":     "i will",
    "you'll":   "you will",
    "he'll":    "he will",
    "she'll":   "she will",
    "they'll":  "they will",
    "i'd":      "i would",
    "you'd":    "you would",
    "he'd":     "he would",
    "she'd":    "she would",
    "they'd":   "they would",
}

In [ ]:
def remove_html(text):
    return BeautifulSoup(text, "html.parser").get_text()


def remove_square_brackets(text):
    return re.sub(r'\[[^\]]*\]', '', text)


def remove_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text)


def expand_contractions(text):
    return " ".join(contractions_dict.get(word, word) for word in text.split())


def remove_special_chars(text):
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

### Text Lemmatization

In [ ]:
def get_wordnet_pos(tag):
    """Map a Penn Treebank POS tag to a WordNet POS constant."""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN  # default


def pos_lemmatizer(tokens):
    """Lemmatize a list of tokens using POS-aware WordNet lemmatization."""
    pos_tags = nltk.pos_tag(tokens)
    return [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags
    ]

### ML Preprocessing

#### Stopword Handling — Preserving Negations

In [ ]:
stop_words = set(stopwords.words('english'))

# Words that reverse sentiment — must NOT be filtered out
negation_words = {
    "no", "nor", "not", "never",
    "don't", "doesn't", "didn't",
    "isn't", "aren't", "wasn't", "weren't",
    "haven't", "hasn't", "hadn't",
    "won't", "wouldn't", "shouldn't", "couldn't",
    "cannot", "can't", "n't",
}

stop_words = stop_words - negation_words

#### ML Preprocessing Pipeline

In [ ]:
def preprocess_review_ml(text):
    """Aggressive preprocessing for TF-IDF + classical ML models."""
    text = remove_html(text)
    text = remove_square_brackets(text)
    text = remove_urls(text)
    text = text.lower()
    text = expand_contractions(text)
    text = remove_special_chars(text)

    tokens = tokenizer.tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = pos_lemmatizer(tokens)

    return " ".join(tokens)

#### Apply ML Preprocessing

In [ ]:
imdb_data['review_ml'] = imdb_data['review'].apply(preprocess_review_ml)

### DL Preprocessing — RNN & LSTM

#### DL Sequence Preprocessing Pipeline

In [ ]:
def preprocess_review_dl(text):
    """
    Light preprocessing for sequence models (RNN / LSTM).
    Stop words are kept so the embedding layer can learn positional context.
    """
    text = remove_html(text)
    text = remove_square_brackets(text)
    text = remove_urls(text)
    text = text.lower()
    text = expand_contractions(text)
    text = remove_special_chars(text)

    tokens = tokenizer.tokenize(text)
    tokens = pos_lemmatizer(tokens)

    return " ".join(tokens)

#### Apply DL Sequence Preprocessing

In [ ]:
imdb_data['review_dl_sequence'] = imdb_data['review'].apply(preprocess_review_dl)

### DL Preprocessing — Transformer

#### Transformer Preprocessing Pipeline

In [ ]:
def preprocess_review_transformer(text):
    
    text = remove_html(text)
    text = remove_square_brackets(text)
    text = remove_urls(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

#### Apply Transformer Preprocessing

In [ ]:
imdb_data['review_dl_transformer'] = imdb_data['review'].apply(preprocess_review_transformer)

### Compare: ML vs DL Sequence vs DL Transformer Preprocessing

| Pipeline | Lowercase | Stopwords removed | Lemmatized | Notes |
|---|---|---|---|---|
| ML | ✅ | ✅ (negations kept) | ✅ | Aggressive — best for TF-IDF |
| DL Sequence | ✅ | ❌ | ✅ | Moderate — RNN/LSTM need context words |
| Transformer | ❌ | ❌ | ❌ | Minimal — WordPiece handles the rest |

In [ ]:
print('=== Original ===')
print(imdb_data['review'][0])

print('\n=== ML Preprocessed (aggressive) ===')
print(imdb_data['review_ml'][0])

print('\n=== DL Sequence Preprocessed (light) ===')
print(imdb_data['review_dl_sequence'][0])

print('\n=== DL Transformer Preprocessed (minimal) ===')
print(imdb_data['review_dl_transformer'][0])

=== Original ===
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show 

In [ ]:
imdb_data.head()

,review,sentiment,review_ml,review_dl_sequence,review_dl_transformer
0,One of the other reviewers has mentioned that ...,positive,one reviewer mention watch oz episode hook rig...,one of the other reviewer have mention that af...,One of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production film technique una...,a wonderful little production the filming tech...,A wonderful little production. The filming tec...
2,I thought this was a wonderful way to spend ti...,positive,think wonderful way spend time hot summer week...,i think this be a wonderful way to spend time ...,I thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically family little boy jake think zombie ...,basically there s a family where a little boy ...,Basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter mattei love time money visually stunnin...,petter mattei s love in the time of money be a...,"Petter Mattei's ""Love in the Time of Money"" is..."


### Transform Target Class


1.   negative -> 0
2.   posetive -> 1



In [ ]:
imdb_data['sentiment'] = imdb_data['sentiment'].map({'negative': 0, 'positive': 1})

In [ ]:
imdb_data['sentiment'].value_counts()

,count
sentiment,
1,24884
0,24698


### Data Splitting

In [ ]:
X_ml             = imdb_data['review_ml']
X_dl_sequence    = imdb_data['review_dl_sequence']
X_dl_transformer = imdb_data['review_dl_transformer']

y = imdb_data['sentiment']

In [ ]:
X_train_ml, X_test_ml, y_train, y_test = train_test_split(
    X_ml, y, test_size=0.2, random_state=42, stratify=y
)

X_train_dl_seq, X_test_dl_seq, _, _ = train_test_split(
    X_dl_sequence, y, test_size=0.2, random_state=42, stratify=y
)

X_train_dl_transformer, X_test_dl_transformer, _, _ = train_test_split(
    X_dl_transformer, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(y_train):,} | Test size: {len(y_test):,}')

Train size: 39,665 | Test size: 9,917


## 5. Modeling

### Machine Learning Models

#### Text Preparation — TF-IDF Vectorization

We experiment with two vocabulary sizes to observe the effect on model performance.

In [ ]:
# TF-IDF with 10,000 features
tfidf_10k = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))
X_train_tfidf_10k = tfidf_10k.fit_transform(X_train_ml)
X_test_tfidf_10k  = tfidf_10k.transform(X_test_ml)

# TF-IDF with 50,000 features
tfidf_50k = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2))
X_train_tfidf_50k = tfidf_50k.fit_transform(X_train_ml)
X_test_tfidf_50k  = tfidf_50k.transform(X_test_ml)

In [ ]:
print(f'TF-IDF 10k — train: {X_train_tfidf_10k.shape} | test: {X_test_tfidf_10k.shape}')
print(f'TF-IDF 50k — train: {X_train_tfidf_50k.shape} | test: {X_test_tfidf_50k.shape}')

TF-IDF 10k — train: (39665, 10000) | test: (9917, 10000)
TF-IDF 50k — train: (39665, 50000) | test: (9917, 50000)


#### 1. Logistic Regression

##### Training

In [ ]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_tfidf_10k, y_train)
y_pred_lr_10k = lr.predict(X_test_tfidf_10k)

lr.fit(X_train_tfidf_50k, y_train)
y_pred_lr_50k = lr.predict(X_test_tfidf_50k)

##### Evaluation — 10k Features

In [ ]:
print(classification_report(y_test, y_pred_lr_10k))
print(confusion_matrix(y_test, y_pred_lr_10k))

              precision    recall  f1-score   support

           0       0.91      0.89      0.90      4940
           1       0.89      0.91      0.90      4977

    accuracy                           0.90      9917
   macro avg       0.90      0.90      0.90      9917
weighted avg       0.90      0.90      0.90      9917

[[4377  563]
 [ 459 4518]]


##### Evaluation — 50k Features

In [ ]:
print(classification_report(y_test, y_pred_lr_50k))
print(confusion_matrix(y_test, y_pred_lr_50k))

              precision    recall  f1-score   support

           0       0.91      0.89      0.90      4940
           1       0.89      0.91      0.90      4977

    accuracy                           0.90      9917
   macro avg       0.90      0.90      0.90      9917
weighted avg       0.90      0.90      0.90      9917

[[4401  539]
 [ 442 4535]]


#### 2. Linear SVC

##### Training

In [ ]:
lsvc = LinearSVC()

lsvc.fit(X_train_tfidf_10k, y_train)
y_pred_lsvc_10k = lsvc.predict(X_test_tfidf_10k)

lsvc.fit(X_train_tfidf_50k, y_train)
y_pred_lsvc_50k = lsvc.predict(X_test_tfidf_50k)

##### Evaluation — 10k Features

In [ ]:
print(classification_report(y_test, y_pred_lsvc_10k))
print(confusion_matrix(y_test, y_pred_lsvc_10k))

              precision    recall  f1-score   support

           0       0.89      0.89      0.89      4940
           1       0.89      0.89      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917

[[4404  536]
 [ 524 4453]]


##### Evaluation — 50k Features

In [ ]:
print(classification_report(y_test, y_pred_lsvc_50k))
print(confusion_matrix(y_test, y_pred_lsvc_50k))

              precision    recall  f1-score   support

           0       0.91      0.90      0.90      4940
           1       0.90      0.91      0.91      4977

    accuracy                           0.91      9917
   macro avg       0.91      0.91      0.91      9917
weighted avg       0.91      0.91      0.91      9917

[[4438  502]
 [ 437 4540]]


#### 3. Naive Bayes

##### Training

In [ ]:
nb = MultinomialNB()

nb.fit(X_train_tfidf_10k, y_train)
y_pred_nb_10k = nb.predict(X_test_tfidf_10k)

nb.fit(X_train_tfidf_50k, y_train)
y_pred_nb_50k = nb.predict(X_test_tfidf_50k)

##### Evaluation — 10k Features

In [ ]:
print(classification_report(y_test, y_pred_nb_10k))
print(confusion_matrix(y_test, y_pred_nb_10k))

              precision    recall  f1-score   support

           0       0.88      0.85      0.86      4940
           1       0.86      0.88      0.87      4977

    accuracy                           0.87      9917
   macro avg       0.87      0.87      0.87      9917
weighted avg       0.87      0.87      0.87      9917

[[4215  725]
 [ 591 4386]]


##### Evaluation — 50k Features

In [ ]:
print(classification_report(y_test, y_pred_nb_50k))
print(confusion_matrix(y_test, y_pred_nb_50k))

              precision    recall  f1-score   support

           0       0.89      0.88      0.88      4940
           1       0.88      0.89      0.88      4977

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917

[[4329  611]
 [ 548 4429]]


#### Save Best ML Model (Linear_SVC) & tfidf_50k

In [ ]:
import os
import joblib

os.makedirs("saved_models/ml", exist_ok=True)

# Save LinearSVC model
joblib.dump(lsvc, "saved_models/ml/linear_svc_model.joblib")

# Save TF-IDF Vectorizer
joblib.dump(tfidf_50k, "saved_models/ml/tfidf_50k_vectorizer.joblib")

['saved_models/ml/tfidf_50k_vectorizer.joblib']

### Deep Learning Models — RNN & LSTM

#### Text Preparation for Sequence Models

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, SimpleRNN, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [ ]:
# Hyperparameters
vocab_size    = 50000
embedding_dim = 64

In [ ]:
# Choose max_length based on the DL sequence training data percentiles
lengths = [len(text.split()) for text in X_train_dl_seq]

print('90th percentile:', np.percentile(lengths, 90))
print('95th percentile:', np.percentile(lengths, 95))
print('99th percentile:', np.percentile(lengths, 99))

90th percentile: 456.0
95th percentile: 597.0
99th percentile: 920.0


In [ ]:
max_length = 580  # covers ~95th percentile of review lengths

In [ ]:
tokenizer_dl = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer_dl.fit_on_texts(X_train_dl_seq)

In [ ]:
# Convert text to integer sequences
X_train_seq = tokenizer_dl.texts_to_sequences(X_train_dl_seq)
X_test_seq  = tokenizer_dl.texts_to_sequences(X_test_dl_seq)

In [ ]:
# Pad / truncate all sequences to max_length
X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_length, padding='post', truncating='post')

print('Train padded shape:', X_train_pad.shape)
print('Test  padded shape:', X_test_pad.shape)

Train padded shape: (39665, 580)
Test  padded shape: (9917, 580)


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=3,
    verbose=1,
    restore_best_weights=True,
)

#### 1. Simple RNN Model

In [ ]:
model_rnn = Sequential([
    Input(shape=(max_length,)),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),
    SimpleRNN(64, return_sequences=False),
    Dense(16, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='sigmoid'),
])

model_rnn.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy'],
)
model_rnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 580, 64)        │     3,200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,209,313 (12.24 MB)

 Trainable params: 3,209,313 (12.24 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_rnn = model_rnn.fit(
    X_train_pad, y_train,
    epochs=15,
    validation_data=(X_test_pad, y_test),
    batch_size=64,
    callbacks=[early_stop],
)

Epoch 1/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 71ms/step - accuracy: 0.6457 - loss: 0.6167 - val_accuracy: 0.5389 - val_loss: 0.6723
Epoch 2/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.7121 - loss: 0.5565 - val_accuracy: 0.6720 - val_loss: 0.5955
Epoch 3/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.7703 - loss: 0.4825 - val_accuracy: 0.7637 - val_loss: 0.5051
Epoch 4/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.7490 - loss: 0.5103 - val_accuracy: 0.6514 - val_loss: 0.6292
Epoch 5/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.7876 - loss: 0.4576 - val_accuracy: 0.7728 - val_loss: 0.5039
Epoch 6/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 51ms/step - accuracy: 0.8327 - loss: 0.3871 - val_accuracy: 0.7895 - val_loss: 0.4687
Epoch 7/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.8866 - loss: 0.2879 - val_accuracy: 0.8262 - val_loss: 0.4252
Epoch 8/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.8602 - loss: 0.3503 - 

In [ ]:
y_pred_rnn        = model_rnn.predict(X_test_pad)
y_pred_rnn_binary = (y_pred_rnn > 0.5).astype(int)

print(classification_report(y_test, y_pred_rnn_binary))
print(confusion_matrix(y_test, y_pred_rnn_binary))

310/310 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step
              precision    recall  f1-score   support

           0       0.80      0.86      0.83      4940
           1       0.85      0.79      0.82      4977

    accuracy                           0.83      9917
   macro avg       0.83      0.83      0.83      9917
weighted avg       0.83      0.83      0.83      9917

[[4267  673]
 [1051 3926]]


#### 2. LSTM Model

In [ ]:
model_lstm = Sequential([
    Input(shape=(max_length,)),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),
    LSTM(64),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid'),
])

model_lstm.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0001),
    metrics=['accuracy'],
)
model_lstm.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 580, 64)        │     3,200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,235,137 (12.34 MB)

 Trainable params: 3,235,137 (12.34 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Dropout(0.3)
history_lstm = model_lstm.fit(
    X_train_pad, y_train,
    epochs=10,
    validation_data=(X_test_pad, y_test),
    batch_size=64,
    callbacks=[early_stop],
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.7027 - loss: 0.5730 - val_accuracy: 0.8514 - val_loss: 0.3779
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 37s 28ms/step - accuracy: 0.8884 - loss: 0.3025 - val_accuracy: 0.8907 - val_loss: 0.2775
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9222 - loss: 0.2170 - val_accuracy: 0.8848 - val_loss: 0.2779
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.9423 - loss: 0.1719 - val_accuracy: 0.8989 - val_loss: 0.2648
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9550 - loss: 0.1384 - val_accuracy: 0.8949 - val_loss: 0.3090
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.9638 - loss: 0.1126 - val_accuracy: 0.8956 - val_loss: 0.3004
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.9718 - loss: 0.0928 - val_accuracy: 0.8871 - val_loss: 0.3009
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 4.


In [ ]:
y_pred_lstm        = model_lstm.predict(X_test_pad)
y_pred_lstm_binary = (y_pred_lstm > 0.5).astype(int)

print(classification_report(y_test, y_pred_lstm_binary))
print(confusion_matrix(y_test, y_pred_lstm_binary))

310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step
              precision    recall  f1-score   support

           0       0.90      0.89      0.90      4940
           1       0.89      0.91      0.90      4977

    accuracy                           0.90      9917
   macro avg       0.90      0.90      0.90      9917
weighted avg       0.90      0.90      0.90      9917

[[4401  539]
 [ 464 4513]]


#### Save Best DL Model (LSTM)

In [ ]:
import os
import joblib
import json

os.makedirs("saved_models/dl_lstm", exist_ok=True)

# Save Keras LSTM model
model_lstm.save("saved_models/dl_lstm/lstm_sentiment_model.keras")

# Save tokenizer
joblib.dump(tokenizer_dl, "saved_models/dl_lstm/tokenizer_dl.joblib")

dl_config = {
    "vocab_size": vocab_size,
    "embedding_dim": embedding_dim,
    "max_length": max_length
}

with open("saved_models/dl_lstm/dl_config.json", "w") as f:
    json.dump(dl_config, f)

### Transformer Model — DistilBERT Fine-Tuning

**Fine-Tuning** `distilbert-base-uncased`

> Transformers use their own subword tokenizer (WordPiece)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback

In [ ]:
print(f'Train size : {len(X_train_dl_transformer):,}')
print(f'Test size  : {len(X_test_dl_transformer):,}')
print(f'\nSample (transformer):\n{X_train_dl_transformer[5]}')

Train size : 39,665
Test size  : 9,917

Sample (transformer):
Probably my all-time favorite movie, a story of selflessness, sacrifice and dedication to a noble cause, but it's not preachy or boring. It just never gets old, despite my having seen it some 15 or more times in the last 25 years. Paul Lukas' performance brings tears to my eyes, and Bette Davis, in one of her very few truly sympathetic roles, is a delight. The kids are, as grandma says, more like "dressed-up midgets" than children, but that only makes them more fun to watch. And the mother's slow awakening to what's happening in the world and under her own roof is believable and startling. If I had a dozen thumbs, they'd all be "up" for this movie.


In [ ]:
model_name = "distilbert-base-uncased"

tokenizer_bert = AutoTokenizer.from_pretrained(model_name)

In [ ]:
train_ds = Dataset.from_dict({
    "text": list(X_train_dl_transformer),
    "label": list(y_train)
})

test_ds = Dataset.from_dict({
    "text": list(X_test_dl_transformer),
    "label": list(y_test)
})

In [ ]:
def tokenize(batch):
    return tokenizer_bert(batch["text"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [ ]:
print(bert_model.config.dim)

768


In [ ]:
args = TrainingArguments(
    output_dir="./sentiment-distilbert",

    num_train_epochs=8,

    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,

    learning_rate=2e-5,
    weight_decay=0.05,
    warmup_steps=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),

    logging_steps=100,
    seed=42
)

trainer = Trainer(
    model=bert_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=DataCollatorWithPadding(tokenizer_bert),
    compute_metrics=compute_metrics,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.265368,0.244251,0.903398,0.906755
2,0.184245,0.251855,0.906222,0.911361
3,0.133939,0.244444,0.916810,0.918759


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1860, training_loss=0.22937327764367543, metrics={'train_runtime': 873.3183, 'train_samples_per_second': 363.35, 'train_steps_per_second': 5.679, 'total_flos': 7881479051535360.0, 'train_loss': 0.22937327764367543, 'epoch': 3.0})

In [ ]:
transformer_preds = trainer.predict(test_ds).predictions
y_pred_transformer = np.argmax(transformer_preds, axis=1)

print(classification_report(y_test, y_pred_transformer))

              precision    recall  f1-score   support

           0       0.93      0.87      0.90      4940
           1       0.88      0.94      0.91      4977

    accuracy                           0.90      9917
   macro avg       0.91      0.90      0.90      9917
weighted avg       0.91      0.90      0.90      9917



In [ ]:
print(confusion_matrix(y_test, y_pred_transformer))

[[4301  639]
 [ 319 4658]]


#### Save Transformer

In [ ]:
import os

transformer_path = "saved_models/transformer_distilbert"

os.makedirs(transformer_path, exist_ok=True)

trainer.save_model(transformer_path)

# Save tokenizer
tokenizer_bert.save_pretrained(transformer_path)